In [47]:
import pandas as pd

In [48]:
df = pd.read_csv("../data/processed/all_sales.csv")
df["date"] = pd.to_datetime(df["date"])

df.head()


,date,sku,item_name,unit_upc,case_upc,quantity,unit_price,total_sales
0,2024-01-02,13353,Natural Light 2x 15 Pack (12 oz Cans),18200000317,18200200779,5,20.35,101.75
1,2024-01-02,15032,Michelob Ultra Infusions Lime and Prickly Pear...,18200007941,18200200199,1,31.50,31.50
2,2024-01-02,15800,Budweiser 30 Pack (12 oz Cans),18200000164,18200110306,1,29.00,29.00
3,2024-01-02,17100,Natural Light 30 Pack (12 oz Cans),18200000317,18200150302,1,21.15,21.15
4,2024-01-02,18048,Michelob Ultra 15 Pack (25 oz Cans),18200250101,18200960116,1,39.75,39.75


In [49]:
df["quantity"].describe()

count    2620.000000
mean        2.305725
std         2.129101
min        -7.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        15.000000
Name: quantity, dtype: float64

In [50]:
df["quantity_clean"] = df["quantity"].clip(lower=0)

In [51]:
df["quantity_clean"].describe()

count    2620.000000
mean        2.327863
std         2.090850
min         0.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        15.000000
Name: quantity_clean, dtype: float64

In [52]:
df = df.sort_values(["item_name", "date"])

In [53]:
df["lag_1"] = df.groupby("item_name")["quantity_clean"].shift(1)

In [54]:
df[["item_name", "date", "quantity_clean", "lag_1"]].head(10)

,item_name,date,quantity_clean,lag_1
1917,Arizona Hard Green Tea 12 Pack (22 oz Cans),2025-08-12,1,NaN
1918,Arizona Hard Peach Tea 12 Pack (22 oz Cans),2025-08-12,1,NaN
1028,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-11-12,1,NaN
1070,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-11-26,1,1.0
1095,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-12-03,1,1.0
1483,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2025-04-15,1,1.0
1692,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2025-06-10,1,1.0
1774,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2025-07-01,1,1.0
2022,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2025-09-09,1,1.0
2151,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2025-10-21,1,1.0


In [55]:
df["lag_2"] = df.groupby("item_name")["quantity_clean"].shift(2)
df["lag_4"] = df.groupby("item_name")["quantity_clean"].shift(4)

In [56]:
df[["item_name", "date", "quantity_clean", "lag_1", "lag_2", "lag_4"]].head(98)

,item_name,date,quantity_clean,lag_1,lag_2,lag_4
1917,Arizona Hard Green Tea 12 Pack (22 oz Cans),2025-08-12,1,NaN,NaN,NaN
1918,Arizona Hard Peach Tea 12 Pack (22 oz Cans),2025-08-12,1,NaN,NaN,NaN
1028,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-11-12,1,NaN,NaN,NaN
1070,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-11-26,1,1.0,NaN,NaN
1095,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-12-03,1,1.0,1.0,NaN
...,...,...,...,...,...,...
416,Bud Light 18 Pack (12 oz Cans),2024-05-21,6,5.0,6.0,6.0
444,Bud Light 18 Pack (12 oz Cans),2024-05-28,8,6.0,5.0,3.0
471,Bud Light 18 Pack (12 oz Cans),2024-06-04,7,8.0,6.0,6.0
495,Bud Light 18 Pack (12 oz Cans),2024-06-11,12,7.0,8.0,5.0


In [57]:
#rolling mean for trend
# for each product take last 4 weeks excluding current and averages them to get the trend
df["rolling_mean_4"] = (
    df.groupby("item_name")["quantity_clean"]
    .shift(1)
    .rolling(window=4)
    .mean()
)

In [58]:
# rolling std will tell us how stable the demand is and will help with predicting uncertainty
df["rolling_std_4"] = (
    df.groupby("item_name")["quantity_clean"]
    .shift(1)
    .rolling(window=4)
    .std()
)

In [59]:
#check

df[[
    "item_name",
    "date",
    "quantity_clean",
    "lag_1",
    "rolling_mean_4",
    "rolling_std_4"
]].head(500)

,item_name,date,quantity_clean,lag_1,rolling_mean_4,rolling_std_4
1917,Arizona Hard Green Tea 12 Pack (22 oz Cans),2025-08-12,1,NaN,NaN,NaN
1918,Arizona Hard Peach Tea 12 Pack (22 oz Cans),2025-08-12,1,NaN,NaN,NaN
1028,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-11-12,1,NaN,NaN,NaN
1070,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-11-26,1,1.0,NaN,NaN
1095,Beatbox Blue Razzberry Malt 12x 500 ml Carton,2024-12-03,1,1.0,NaN,NaN
...,...,...,...,...,...,...
2603,Bud Light 30 Pack (12 oz Cans),2026-03-17,2,3.0,3.75,1.707825
316,Bud Light 3x 8 Pack (16 oz Aluminum Bottles),2024-04-16,1,NaN,NaN,NaN
387,Bud Light 3x 8 Pack (16 oz Aluminum Bottles),2024-05-07,1,1.0,NaN,NaN
605,Bud Light 3x 8 Pack (16 oz Aluminum Bottles),2024-07-02,2,1.0,NaN,NaN


In [60]:
df = pd.read_csv("../data/processed/weekly_orders_clean.csv")
df["date"] = pd.to_datetime(df["date"])

In [61]:
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

In [62]:
df[["date", "week", "month", "quarter"]].head(10)

,date,week,month,quarter
0,2024-01-02,1,1,1
1,2024-01-02,1,1,1
2,2024-01-02,1,1,1
3,2024-01-02,1,1,1
4,2024-01-02,1,1,1
5,2024-01-02,1,1,1
6,2024-01-02,1,1,1
7,2024-01-02,1,1,1
8,2024-01-02,1,1,1
9,2024-01-02,1,1,1


In [63]:
df.groupby("date")["item_name"].nunique().head()

date
2024-01-02    98
2024-01-09    98
2024-01-16    98
2024-01-24    98
2024-01-30    98
Name: item_name, dtype: int64

In [64]:
df.groupby("date").size().head()

date
2024-01-02    98
2024-01-09    98
2024-01-16    98
2024-01-24    98
2024-01-30    98
dtype: int64

In [65]:
df[df["date"] == "2024-01-02"].shape

(98, 6)

In [66]:
df["lag_1"] = df.groupby("item_name")["quantity_clean"].shift(1)
df["lag_2"] = df.groupby("item_name")["quantity_clean"].shift(2)
df["lag_4"] = df.groupby("item_name")["quantity_clean"].shift(4)

In [67]:
df[["item_name", "date", "quantity_clean", "lag_1", "lag_2", "lag_4"]].head(500)

,item_name,date,quantity_clean,lag_1,lag_2,lag_4
0,Natural Light 2x 15 Pack (12 oz Cans),2024-01-02,5.0,NaN,NaN,NaN
1,Shock Top Belgian White 4x 6 Pack (12 oz Bottles),2024-01-02,1.0,NaN,NaN,NaN
2,Michelob Ultra 4x 6 Pack (12 oz Bottles),2024-01-02,1.0,NaN,NaN,NaN
3,Michelob Ultra 18 Pack (12 oz Bottles),2024-01-02,1.0,NaN,NaN,NaN
4,Bud Light 6x 4 Pack (16 oz Cans),2024-01-02,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...
495,Natural Light 4x 6 Pack (16 oz Cans),2024-02-06,2.0,1.0,0.0,2.0
496,Bud Light 3x 8 Pack (16 oz Cans),2024-02-06,0.0,1.0,2.0,1.0
497,Bud Light 20 Pack (12 oz Bottles),2024-02-06,6.0,0.0,1.0,0.0
498,Budweiser Chelada with Clamato 15 Pack (25 oz ...,2024-02-06,0.0,0.0,0.0,0.0


In [68]:
g = df.groupby("item_name")["quantity_clean"]

df["rolling_mean_4"] = g.shift(1).rolling(4).mean()

df["rolling_std_4"] = g.shift(1).rolling(4).std()

In [70]:
df[[
    "item_name", "date", "quantity_clean",
    "rolling_mean_4",
    "rolling_std_4"
]].head(15)

,item_name,date,quantity_clean,rolling_mean_4,rolling_std_4
0,Natural Light 2x 15 Pack (12 oz Cans),2024-01-02,5.0,NaN,NaN
1,Shock Top Belgian White 4x 6 Pack (12 oz Bottles),2024-01-02,1.0,NaN,NaN
2,Michelob Ultra 4x 6 Pack (12 oz Bottles),2024-01-02,1.0,NaN,NaN
3,Michelob Ultra 18 Pack (12 oz Bottles),2024-01-02,1.0,NaN,NaN
4,Bud Light 6x 4 Pack (16 oz Cans),2024-01-02,1.0,NaN,NaN
5,Natural Light 4x 6 Pack (16 oz Cans),2024-01-02,2.0,NaN,NaN
6,Bud Light 3x 8 Pack (16 oz Cans),2024-01-02,1.0,NaN,NaN
7,Bud Light 20 Pack (12 oz Bottles),2024-01-02,6.0,NaN,NaN
8,Budweiser Chelada with Clamato 15 Pack (25 oz ...,2024-01-02,1.0,NaN,NaN
9,Michelob Ultra 15 Pack (25 oz Cans),2024-01-02,1.0,NaN,NaN


In [73]:
# drop rows with NaN values caused by the lag rolling features
df_clean = df.dropna().reset_index(drop=True)

# export the engineered features dataset
df_clean.to_csv("../data/processed/featured_sales.csv", index=False)
print("features successfully exported to featured_sales.csv")


features successfully exported to featured_sales.csv
